In [ ]:
import numpy as np
import sklearn.decomposition as dp
import pickle
import sys,os
import numpy.random as rand
from sklearn.linear_model import LogisticRegression as LR
from sklearn.metrics import auc,roc_curve,roc_auc_score
from sklearn.metrics import average_precision_score,precision_recall_curve
from sklearn.utils.random import sample_without_replacement

sys.path.append('/home/austin/DataAnalysis')
from data_tools import load_data


In [ ]:
def extract_data(fnm):
    power,coherence,granger,labels = load_data(fnm,fBounds=(1,56),
                        feature_list=['power','coherence','granger'])
    myLabel = labels['windows']
    mouse = np.asarray(myLabel['mouse'])
    expDate = np.asarray(myLabel['expDate'])
    behavior = np.asarray(myLabel['behavior'])
    time = np.asarray(myLabel['time'])
    condition = np.asarray(myLabel['condition'])
    
    # Select indexes of interest
    indx_pos = (behavior==1)&(condition==4)
    indx_neg1 = (behavior==2)&(condition==4)
    indx_neg2 = (behavior==2)&(condition==6)
    indx_neg3 = (behavior==2)&(condition==8)
    indx_neg = indx_neg1|indx_neg2|indx_neg3
    indx_tot = indx_neg|indx_pos
    
    y = np.zeros(len(time))
    y[indx_pos] = 1
    
    #Now subselect the windows
    power = power[indx_tot]
    coherence = coherence[indx_tot]
    granger = granger[indx_tot]
    mouse = mouse[indx_tot]
    expDate = expDate[indx_tot]
    behavior = behavior[indx_tot]
    time = time[indx_tot]
    condition = condition[indx_tot]
    y = y[indx_tot]

    #Combine the features
    granger = np.exp(granger)
    granger[granger>10] = 10
    power = power*10
    power[power>6] = 6
    X = np.hstack((power,coherence,granger))
    
    #Store in a nice neat dictionary
    md = {}
    md['X'] = X
    md['y'] = y
    md['mouse'] = mouse
    md['expDate'] = expDate
    md['behavior'] = behavior
    md['time'] = time
    md['condition'] = condition
    
    return md

In [ ]:
def extract_data_behaviornon1(fnm):
    power,coherence,granger,labels = load_data(fnm,fBounds=(1,56),
                        feature_list=['power','coherence','granger'])
    myLabel = labels['windows']
    mouse = np.asarray(myLabel['mouse'])
    expDate = np.asarray(myLabel['expDate'])
    behavior = np.asarray(myLabel['behaviornon1'])
    time = np.asarray(myLabel['time'])
    condition = np.asarray(myLabel['condition'])
    
    # Select indexes of interest
    indx_pos = (behavior==1)&(condition==4)
    indx_neg1 = (behavior==2)&(condition==4)
    indx_neg2 = (behavior==2)&(condition==6)
    indx_neg3 = (behavior==2)&(condition==8)
    indx_neg = indx_neg1|indx_neg2|indx_neg3
    indx_tot = indx_neg|indx_pos
    
    y = np.zeros(len(time))
    y[indx_pos] = 1
    
    #Now subselect the windows
    power = power[indx_tot]
    coherence = coherence[indx_tot]
    granger = granger[indx_tot]
    mouse = mouse[indx_tot]
    expDate = expDate[indx_tot]
    behavior = behavior[indx_tot]
    time = time[indx_tot]
    condition = condition[indx_tot]
    y = y[indx_tot]

    #Combine the features
    granger = np.exp(granger)
    granger[granger>10] = 10
    power = power*10
    power[power>6] = 6
    X = np.hstack((power,coherence,granger))
    
    #Store in a nice neat dictionary
    md = {}
    md['X'] = X
    md['y'] = y
    md['mouse'] = mouse
    md['expDate'] = expDate
    md['behavior'] = behavior
    md['time'] = time
    md['condition'] = condition
    
    return md

In [ ]:
dirname = '/media/austin/ThickBoy__1/DataAgression_Granger2/'
data_clbaseline_2021 = extract_data(dirname+'CL_baseline_2021_1.mat')
print(np.unique(data_clbaseline_2021['mouse']))
print(data_clbaseline_2021['X'].shape)

In [ ]:
data_clbaseline_2021_2 = extract_data(dirname+'CL_baseline_2021_2.mat')
print(np.unique(data_clbaseline_2021_2['mouse']))
print(data_clbaseline_2021_2['X'].shape)

In [ ]:
data_clbaseline_validate3 = extract_data(dirname+'CL_baseline_all_validate3.mat')
print(np.unique(data_clbaseline_validate3['mouse']))
print(data_clbaseline_validate3['X'].shape)

In [ ]:
data_aggresion_sub = extract_data_behaviornon1(dirname+'Aggression_sub_12.mat')
print(np.unique(data_aggresion_sub['mouse']))
print(data_aggresion_sub['X'].shape)

In [ ]:
model_dict = pickle.load(open('Unbalanced_Elastic_12_enc_1.0.p','rb'))

In [ ]:
A_enc = model_dict['A_enc']
B_enc = model_dict['B_enc']

In [ ]:
def get_test_set(mouse,mice_test):
    N_test = len(mice_test)
    idxs = np.zeros(len(mouse))
    for i in range(N_test):
        idxs[mouse==mice_test[i]] = 1
    return idxs

In [ ]:
def softplus_np(x):
    ''' 
    Computes the softplus activation of x in a numerically stable way
    y = np.log(np.exp(y) + 1)

    Parameters 
    ----------
    x : np.array
        Original array
    Returns
    -------
    y : np.array
        Transformed array
    '''
    y = np.log(1+np.exp(-np.abs(x))) + np.maximum(x,0)
    return y


In [ ]:
def project(myDict,A_enc,B_enc):
    myDict['S'] = softplus_np(np.dot(myDict['X'],A_enc)+B_enc)

In [ ]:
project(data_aggresion_sub,A_enc,B_enc)
project(data_clbaseline_validate3,A_enc,B_enc)
project(data_clbaseline_2021_2,A_enc,B_enc)
project(data_clbaseline_2021,A_enc,B_enc)

In [ ]:
def evaluate_auc(myDict,factor_number,sign):
    y_hat = myDict['S'][:,factor_number]*sign
    mice = np.unique(myDict['mouse'])
    results_dict = {}
    for i in range(len(mice)):
        ids = myDict['mouse']==mice[i]
        try:
            results_dict[mice[i]] = roc_auc_score(myDict['y'][ids],y_hat[ids])
        except:
            results_dict[mice[i]] = -1
    return results_dict

def print_results(myDict):
    for key in myDict.keys():
        print(key,myDict[key])
        
def get_mean_std_results(myDict):
    nmice = len(key)

In [ ]:
mice_test_agg = ['Mouse048','Mouse7980','Mouse7998']
mice_test_val3 = ['Mouse5564','Mouse5565','Mouse5566','Mouse5567']

## Let's get results of first factor

In [ ]:
results_aggresion_sub_1 = evaluate_auc(data_aggresion_sub,0,-1)
results_clbaseline_validate3_1 = evaluate_auc(data_clbaseline_validate3,0,-1)
results_clbaseline_2021_2_1 = evaluate_auc(data_clbaseline_2021_2,0,-1)
results_clbaseline_2021_1 = evaluate_auc(data_clbaseline_2021,0,-1)

In [ ]:
print_results(results_aggresion_sub_1)

In [ ]:
print_results(results_clbaseline_validate3_1)

In [ ]:
print_results(results_clbaseline_2021_2_1)

In [ ]:
print_results(results_clbaseline_2021_1)

## Let's get results of factor 6

In [ ]:
fnum = 5
sgn = 1.0
results_aggresion_sub_6 = evaluate_auc(data_aggresion_sub,fnum,sgn)
results_clbaseline_validate3_6 = evaluate_auc(data_clbaseline_validate3,fnum,sgn)
results_clbaseline_2021_2_6 = evaluate_auc(data_clbaseline_2021_2,fnum,sgn)
results_clbaseline_2021_6 = evaluate_auc(data_clbaseline_2021,fnum,sgn)

In [ ]:
print_results(results_aggresion_sub_6)
print('-----')
print_results(results_clbaseline_validate3_6)
print('>>>>>>')
print_results(results_clbaseline_2021_2_6)
print_results(results_clbaseline_2021_6)

### Combined model 

In [ ]:
mice_test_agg = ['Mouse048','Mouse7980','Mouse7998']
mice_test_val3 = ['Mouse5564','Mouse5565','Mouse5566','Mouse5567']
def train_model(dict_one,dict_two,test_mice1,test_mice2,factor1,factor2):
    S1 = dict_one['S']
    S2 = dict_two['S']
    test_1_idx = get_test_set(dict_one['mouse'],test_mice1)
    test_2_idx = get_test_set(dict_two['mouse'],test_mice2)
    S_train_1 = S1[test_1_idx==0]
    S_train_2 = S2[test_2_idx==0]
    y_train_1 = dict_one['y'][test_1_idx==0]
    y_train_2 = dict_two['y'][test_2_idx==0]
    
    S_tr = np.vstack((S_train_1,S_train_2))
    y_tr = np.concatenate((y_train_1,y_train_2))
    
    S_tr_sub = S_tr[:,[factor1,factor2]]
    
    model_lr = LR()
    model_lr.fit(S_tr_sub,y_tr)
    return model_lr

In [ ]:
model_logreg = train_model(data_aggresion_sub,data_clbaseline_validate3,mice_test_agg,mice_test_val3,0,5)
print(model_logreg.coef_)

In [ ]:
def evaluate_auc_model(myDict,factor_number1,factor_number2,model):
    S_sub = myDict['S'][:,[factor_number1,factor_number2]]
    y_hat = model.decision_function(S_sub)
    mice = np.unique(myDict['mouse'])
    results_dict = {}
    for i in range(len(mice)):
        ids = myDict['mouse']==mice[i]
        try:
            results_dict[mice[i]] = roc_auc_score(myDict['y'][ids],y_hat[ids])
        except:
            results_dict[mice[i]] = -1
    return results_dict

In [ ]:
fn0 = 0
fn1 = 5
results_aggresion_sub_comb = evaluate_auc_model(data_aggresion_sub,fn0,fn1,model_logreg)
results_clbaseline_validate3_comb = evaluate_auc_model(data_clbaseline_validate3,fn0,fn1,model_logreg)
results_clbaseline_2021_2_comb = evaluate_auc_model(data_clbaseline_2021_2,fn0,fn1,model_logreg)
results_clbaseline_2021_comb = evaluate_auc_model(data_clbaseline_2021,fn0,fn1,model_logreg)

In [ ]:
print_results(results_aggresion_sub_comb)
print('-----')
print_results(results_clbaseline_validate3_comb)
print('>>>>>>')
print_results(results_clbaseline_2021_2_comb)
print_results(results_clbaseline_2021_comb)